# Bộ Cân Bằng Âm Thanh Số (FIR Graphic Equalizer)
**Môn:** Xử Lý Tín Hiệu Số | **Nhóm:** [Nhóm X]

## Mục tiêu
- Thiết kế bộ cân bằng âm thanh 10 bands sử dụng bộ lọc **FIR với cửa sổ Hamming**
- Áp dụng equalizer trên file audio `.wav`
- Trực quan hóa: waveform và phổ tần số trước/sau khi cân bằng
- Lưu file âm thanh đã xử lý

## Thông số thiết kế
| Thông số | Giá trị |
|----------|--------|
| Tần số lấy mẫu | 44,100 Hz |
| Số bands | 10 |
| Loại bộ lọc | FIR Bandpass |
| Cửa sổ | Hamming |
| Số taps (N) | 1025 |
| Dải gain | -12 dB đến +12 dB |

## 1. Import thư viện

In [ ]:
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Kiểm tra thư viện
print('NumPy:', np.__version__)
import scipy; print('SciPy:', scipy.__version__)
import soundfile; print('SoundFile:', soundfile.__version__)
print('Tất cả thư viện OK!')

## 2. Cấu hình 10 Bands và Thông số

In [ ]:
# =============================================================
# THÔNG SỐ EQUALIZER
# =============================================================
FS = 44100          # Tần số lấy mẫu (Hz)
NUMTAPS = 1025      # Số tap FIR (phải lẻ cho linear phase)
                    # Độ trễ = (NUMTAPS-1)/2 / FS ≈ 11.6 ms

# 10 bands: (tên, tần số thấp, tần số cao) — đơn vị Hz
BANDS = [
    ('Sub-bass',   20,    60),
    ('Bass',       60,    170),
    ('Low-mid',    170,   310),
    ('Mid',        310,   600),
    ('Upper-mid',  600,   1000),
    ('Presence',   1000,  3000),
    ('Brilliance', 3000,  6000),
    ('Air',        6000,  10000),
    ('High air',   10000, 16000),
    ('Ultra-high', 16000, 20000),
]

# Gain mặc định cho từng band (dB)
# Thay đổi các giá trị này để cân bằng âm thanh
# Dương = tăng (boost), Âm = giảm (cut)
GAINS_DB = [
    0,   # Sub-bass
    0,   # Bass
    0,   # Low-mid
    0,   # Mid
    0,   # Upper-mid
    0,   # Presence
    0,   # Brilliance
    0,   # Air
    0,   # High air
    0,   # Ultra-high
]

print(f'Tần số lấy mẫu: {FS} Hz')
print(f'Tần số Nyquist: {FS//2} Hz')
print(f'Số taps: {NUMTAPS}')
print(f'Độ trễ bộ lọc: {(NUMTAPS-1)/2/FS*1000:.1f} ms')
print(f'Số bands: {len(BANDS)}')
print()
print('Cấu hình bands:')
for i, (name, low, high) in enumerate(BANDS):
    print(f'  Band {i+1:2d}: {name:12s}  {low:6d} – {high:6d} Hz  |  Gain: {GAINS_DB[i]:+5.1f} dB')

## 3. Thiết Kế FIR Bandpass Filter

### Cơ sở lý thuyết
Dùng hàm `scipy.signal.firwin()` với phương pháp **cửa sổ Hamming**:

$$w[n] = 0.54 - 0.46 \cos\left(\frac{2\pi n}{M}\right), \quad 0 \le n \le M$$

Hệ số FIR: $h[n] = h_{ideal}[n] \times w[n]$

Phương trình sai phân:
$$y[n] = \sum_{k=0}^{M} b_k \cdot x[n-k]$$

In [ ]:
def design_fir_bandpass(low_hz, high_hz, fs=FS, numtaps=NUMTAPS):
    """
    Thiết kế bộ lọc FIR bandpass bằng phương pháp cửa sổ Hamming.
    
    Tham số:
        low_hz  : tần số cắt thấp (Hz)
        high_hz : tần số cắt cao (Hz)
        fs      : tần số lấy mẫu (Hz)
        numtaps : số hệ số FIR (phải lẻ)
    
    Trả về:
        b : mảng hệ số FIR, shape=(numtaps,)
    """
    nyquist = fs / 2.0
    
    # Chuẩn hóa về [0, 1] so với Nyquist
    low_norm  = low_hz  / nyquist
    high_norm = high_hz / nyquist
    
    # Giới hạn để tránh lỗi
    low_norm  = max(low_norm,  1e-4)
    high_norm = min(high_norm, 1.0 - 1e-4)
    
    b = signal.firwin(
        numtaps,
        [low_norm, high_norm],
        pass_zero=False,     # Bandpass (không đi qua DC)
        window='hamming'
    )
    return b


# Thiết kế tất cả 10 filters
filters = []
for name, low, high in BANDS:
    b = design_fir_bandpass(low, high)
    filters.append(b)
    print(f'Thiết kế OK: {name:12s} ({low:6d}-{high:6d} Hz)')

print(f'\nTổng: {len(filters)} bộ lọc FIR, mỗi bộ {NUMTAPS} hệ số')

## 4. Vẽ Đáp Ứng Tần Số Của 10 Bands

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# === Đáp ứng biên độ ===
ax1 = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(BANDS)))

for i, (b, (name, low, high)) in enumerate(zip(filters, BANDS)):
    w, h = signal.freqz(b, worN=8192, fs=FS)
    h_db = 20 * np.log10(np.maximum(np.abs(h), 1e-10))
    ax1.plot(w, h_db, color=colors[i], linewidth=1.5,
             label=f'Band {i+1}: {name} ({low}-{high} Hz)')

ax1.set_xscale('log')
ax1.set_xlim([20, 22050])
ax1.set_ylim([-80, 5])
ax1.set_xlabel('Tần số (Hz)', fontsize=12)
ax1.set_ylabel('Biên độ (dB)', fontsize=12)
ax1.set_title('Đáp Ứng Tần Số Của 10 FIR Bandpass Filters (Hamming Window)', fontsize=13)
ax1.axhline(-3, color='gray', linestyle='--', alpha=0.5, label='-3 dB')
ax1.legend(loc='lower right', fontsize=8, ncol=2)
ax1.grid(True, which='both', alpha=0.3)

# Đánh dấu vị trí các bands
for name, low, high in BANDS:
    center = np.sqrt(low * high)  # Tần số trung tâm (log scale)
    ax1.axvline(center, color='gray', alpha=0.15, linewidth=0.8)

# === Đáp ứng pha ===
ax2 = axes[1]
for i, (b, (name, low, high)) in enumerate(zip(filters, BANDS)):
    w, h = signal.freqz(b, worN=8192, fs=FS)
    # Chỉ vẽ pha trong dải thông
    h_db = 20 * np.log10(np.maximum(np.abs(h), 1e-10))
    mask = h_db > -20  # Chỉ dải thông
    phase = np.angle(h, deg=True)
    ax2.plot(w[mask], phase[mask], color=colors[i], linewidth=1.5,
             label=f'{name}')

ax2.set_xscale('log')
ax2.set_xlim([20, 22050])
ax2.set_xlabel('Tần số (Hz)', fontsize=12)
ax2.set_ylabel('Pha (độ)', fontsize=12)
ax2.set_title('Đáp Ứng Pha (trong dải thông)', fontsize=13)
ax2.legend(loc='lower right', fontsize=8, ncol=2)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('frequency_response.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 1: Đáp ứng tần số của 10 FIR bandpass filters')
print('Nhận xét: FIR với cửa sổ Hamming cho đáp ứng pha tuyến tính trong dải thông.')

## 5. Load File Audio

In [ ]:
# ================================================
# ĐỔI ĐƯỜNG DẪN FILE ÂM THANH Ở ĐÂY
# ================================================
AUDIO_FILE = 'audio-equalizer-master/Audio/CantinaBand60.wav'
# AUDIO_FILE = 'test-data/blues/blues.00081.wav'  # Hoặc dùng file từ dataset

def load_audio(file_path, target_fs=FS):
    """
    Load file audio, resample về target_fs nếu cần, chuyển về mono.
    
    Trả về:
        audio : numpy array float32, shape=(N,)
        fs    : tần số lấy mẫu thực tế
    """
    audio, fs = sf.read(file_path, dtype='float32')
    
    # Chuyển stereo → mono (lấy trung bình 2 kênh)
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)
        print(f'Stereo → Mono (trung bình 2 kênh)')
    
    # Resample nếu cần
    if fs != target_fs:
        import librosa
        audio = librosa.resample(audio, orig_sr=fs, target_sr=target_fs)
        fs = target_fs
        print(f'Resampled: {fs} Hz → {target_fs} Hz')
    
    # Chuẩn hóa về [-1, 1]
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    
    return audio, fs


# Load audio
audio_orig, fs = load_audio(AUDIO_FILE)

duration = len(audio_orig) / fs
print(f'File: {AUDIO_FILE}')
print(f'Tần số lấy mẫu: {fs} Hz')
print(f'Số mẫu: {len(audio_orig):,}')
print(f'Thời lượng: {duration:.2f} giây ({duration/60:.1f} phút)')
print(f'Min biên độ: {audio_orig.min():.4f}')
print(f'Max biên độ: {audio_orig.max():.4f}')
print(f'RMS: {np.sqrt(np.mean(audio_orig**2)):.4f}')

## 6. Thiết Lập Gain và Áp Dụng Equalizer

### Công thức áp dụng gain
$$gain_{linear} = 10^{gain_{dB}/20}$$
$$y[n] = \sum_{i=1}^{10} gain_i \cdot (b_i * x)[n]$$

Trong đó $b_i * x$ là tích chập của hệ số FIR band $i$ với tín hiệu vào.

In [ ]:
# ================================================
# THAY ĐỔI GAIN Ở ĐÂY (đơn vị dB)
# Dương = boost (tăng), Âm = cut (giảm)
# Giới hạn: -12 dB đến +12 dB
# ================================================
GAINS_DB = [
    +6,   # Band 1: Sub-bass     (20-60 Hz)     — Tăng bass sâu
    +3,   # Band 2: Bass         (60-170 Hz)    — Tăng nhẹ bass
     0,   # Band 3: Low-mid      (170-310 Hz)
    -3,   # Band 4: Mid          (310-600 Hz)   — Giảm mid hơi "bồng"
     0,   # Band 5: Upper-mid    (600-1000 Hz)
    +3,   # Band 6: Presence     (1000-3000 Hz) — Tăng rõ giọng
    +2,   # Band 7: Brilliance   (3000-6000 Hz)
     0,   # Band 8: Air          (6000-10000 Hz)
    -2,   # Band 9: High air     (10000-16000 Hz)
    -6,   # Band 10: Ultra-high  (16000-20000 Hz) — Giảm noise cao
]

def apply_equalizer(audio, filters, gains_db):
    """
    Áp dụng bộ lọc FIR equalizer.
    
    Quy trình:
    1. Với mỗi band: lọc tín hiệu qua FIR bandpass
    2. Nhân với gain (linear)
    3. Cộng tất cả bands lại
    4. Chuẩn hóa để tránh clip
    """
    output = np.zeros_like(audio)
    band_signals = []
    
    for i, (b, gain_db) in enumerate(zip(filters, gains_db)):
        # Áp dụng FIR filter (giữ trạng thái bộ lọc không cần thiết ở đây vì xử lý 1 file)
        filtered = signal.lfilter(b, [1.0], audio)
        
        # Chuyển gain dB → linear
        gain_linear = 10 ** (gain_db / 20.0)
        
        band_out = filtered * gain_linear
        band_signals.append(band_out)
        output += band_out
    
    # Chuẩn hóa để tránh clipping
    max_val = np.max(np.abs(output))
    if max_val > 0:
        output = output / max_val
    
    return output, band_signals


print('Áp dụng equalizer...')
audio_eq, band_signals = apply_equalizer(audio_orig, filters, GAINS_DB)

print(f'Tín hiệu gốc   — RMS: {np.sqrt(np.mean(audio_orig**2)):.4f}  |  Max: {np.max(np.abs(audio_orig)):.4f}')
print(f'Tín hiệu EQ    — RMS: {np.sqrt(np.mean(audio_eq**2)):.4f}  |  Max: {np.max(np.abs(audio_eq)):.4f}')
print()
print('Cài đặt EQ:')
for i, ((name, low, high), gain_db) in enumerate(zip(BANDS, GAINS_DB)):
    bar = '█' * int(abs(gain_db)) if gain_db != 0 else '─'
    sign = '+' if gain_db > 0 else ''
    print(f'  Band {i+1:2d} {name:12s}: {sign}{gain_db:+5.1f} dB  {bar}')

## 7. Trực Quan Hóa — Waveform (Dạng Sóng Thời Gian)

In [ ]:
# Chỉ vẽ 3 giây đầu để rõ hơn
SHOW_SEC = 3.0
n_show = int(SHOW_SEC * fs)
t = np.arange(n_show) / fs

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('Waveform — Dạng Sóng Thời Gian', fontsize=14, fontweight='bold')

# Tín hiệu gốc
axes[0].plot(t, audio_orig[:n_show], color='steelblue', linewidth=0.8, alpha=0.9)
axes[0].set_title('Tín hiệu GỐC (chưa cân bằng)', fontsize=12)
axes[0].set_ylabel('Biên độ (chuẩn hóa)')
axes[0].set_ylim([-1.1, 1.1])
axes[0].axhline(0, color='k', linewidth=0.5)
rms_orig = np.sqrt(np.mean(audio_orig[:n_show]**2))
axes[0].set_xlabel(f'Thời gian (giây) — RMS = {rms_orig:.4f}')
axes[0].grid(True, alpha=0.3)

# Tín hiệu sau EQ
axes[1].plot(t, audio_eq[:n_show], color='crimson', linewidth=0.8, alpha=0.9)
axes[1].set_title('Tín hiệu SAU cân bằng (EQ)', fontsize=12)
axes[1].set_ylabel('Biên độ (chuẩn hóa)')
axes[1].set_ylim([-1.1, 1.1])
axes[1].axhline(0, color='k', linewidth=0.5)
rms_eq = np.sqrt(np.mean(audio_eq[:n_show]**2))
axes[1].set_xlabel(f'Thời gian (giây) — RMS = {rms_eq:.4f}')
axes[1].grid(True, alpha=0.3)

# Overlay để so sánh
axes[2].plot(t, audio_orig[:n_show], color='steelblue', linewidth=0.8, alpha=0.7, label='Gốc')
axes[2].plot(t, audio_eq[:n_show], color='crimson', linewidth=0.8, alpha=0.7, label='EQ')
axes[2].set_title('So sánh: Gốc vs EQ (overlay)', fontsize=12)
axes[2].set_ylabel('Biên độ (chuẩn hóa)')
axes[2].set_xlabel('Thời gian (giây)')
axes[2].set_ylim([-1.1, 1.1])
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('waveform_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 2: So sánh waveform trước và sau khi cân bằng')

## 8. Trực Quan Hóa — Phổ Tần Số (Frequency Spectrum)

In [ ]:
def compute_spectrum(audio, fs, nperseg=4096):
    """
    Tính phổ công suất trung bình (Power Spectral Density) dùng Welch method.
    Welch method: chia tín hiệu thành nhiều đoạn nhỏ, tính FFT từng đoạn,
    trung bình để giảm nhiễu.
    """
    freqs, psd = signal.welch(audio, fs=fs, nperseg=nperseg,
                               window='hamming', scaling='spectrum')
    # Chuyển về dB
    psd_db = 10 * np.log10(np.maximum(psd, 1e-12))
    return freqs, psd_db


freqs_orig, psd_orig = compute_spectrum(audio_orig, fs)
freqs_eq,   psd_eq   = compute_spectrum(audio_eq, fs)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('Phổ Tần Số — So Sánh Trước và Sau EQ', fontsize=14, fontweight='bold')

# Phổ riêng lẻ
axes[0].plot(freqs_orig, psd_orig, color='steelblue', linewidth=1.5, label='Gốc')
axes[0].plot(freqs_eq,   psd_eq,   color='crimson',   linewidth=1.5, label='EQ', alpha=0.85)
axes[0].set_xscale('log')
axes[0].set_xlim([20, 22050])
axes[0].set_xlabel('Tần số (Hz)', fontsize=12)
axes[0].set_ylabel('Công suất (dB)', fontsize=12)
axes[0].set_title('Power Spectral Density — Gốc vs EQ', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, which='both', alpha=0.3)

# Đánh dấu vị trí các band trên phổ
colors_band = plt.cm.tab10(np.linspace(0, 1, len(BANDS)))
for i, (name, low, high) in enumerate(BANDS):
    center = np.sqrt(low * high)
    axes[0].axvspan(low, high, alpha=0.06, color=colors_band[i])
    if GAINS_DB[i] != 0:
        axes[0].text(center, axes[0].get_ylim()[0] + 2,
                    f'{GAINS_DB[i]:+.0f}dB', ha='center', fontsize=7,
                    color=colors_band[i], rotation=90)

# Hiệu chỉnh EQ (difference)
# Nội suy psd_eq về cùng tần số với psd_orig để trừ
diff = psd_eq - psd_orig
axes[1].fill_between(freqs_orig, diff, 0,
                     where=(diff >= 0), color='green', alpha=0.5, label='Boost')
axes[1].fill_between(freqs_orig, diff, 0,
                     where=(diff < 0), color='red', alpha=0.5, label='Cut')
axes[1].plot(freqs_orig, diff, color='black', linewidth=0.8)
axes[1].axhline(0, color='k', linewidth=1)
axes[1].set_xscale('log')
axes[1].set_xlim([20, 22050])
axes[1].set_xlabel('Tần số (Hz)', fontsize=12)
axes[1].set_ylabel('Hiệu chỉnh (dB)', fontsize=12)
axes[1].set_title('EQ Curve (EQ - Gốc)', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('spectrum_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 3: Phổ tần số trước và sau EQ')
print('Vùng xanh = boost (tăng), vùng đỏ = cut (giảm)')

## 9. Phân Tích Band-by-Band

In [ ]:
# Tính năng lượng mỗi band trước và sau EQ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Năng Lượng Mỗi Band — Trước vs Sau EQ', fontsize=13, fontweight='bold')

band_names = [f'B{i+1}\n{name[:8]}' for i, (name, _, _) in enumerate(BANDS)]
energy_orig = []
energy_eq_list = []

for i, b in enumerate(filters):
    # Năng lượng band gốc
    orig_band = signal.lfilter(b, [1.0], audio_orig)
    en_orig = np.sqrt(np.mean(orig_band**2))
    energy_orig.append(en_orig)
    
    # Năng lượng band sau EQ
    gain_linear = 10 ** (GAINS_DB[i] / 20.0)
    en_eq = en_orig * gain_linear
    energy_eq_list.append(en_eq)

x = np.arange(len(BANDS))
width = 0.35

# Bar chart
axes[0].bar(x - width/2, energy_orig, width, label='Gốc', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, energy_eq_list, width, label='EQ', color='crimson', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(band_names, fontsize=8)
axes[0].set_ylabel('RMS Năng lượng')
axes[0].set_title('Năng lượng theo band (RMS)')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

# Gain visualization
colors_gain = ['green' if g >= 0 else 'red' for g in GAINS_DB]
bars = axes[1].bar(x, GAINS_DB, color=colors_gain, alpha=0.8, edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(band_names, fontsize=8)
axes[1].set_ylabel('Gain (dB)')
axes[1].set_title('Cài đặt EQ (Gain mỗi band)')
axes[1].axhline(0, color='k', linewidth=1)
axes[1].set_ylim([-15, 15])
axes[1].grid(True, axis='y', alpha=0.3)

# Gán nhãn giá trị
for bar, val in zip(bars, GAINS_DB):
    if val != 0:
        axes[1].text(bar.get_x() + bar.get_width()/2,
                    val + (0.3 if val > 0 else -1),
                    f'{val:+.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('band_energy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 4: Năng lượng từng band và cài đặt gain EQ')

## 10. Lưu File Audio Đầu Ra

In [ ]:
OUTPUT_FILE = 'audio_equalized.wav'

# Chuyển về int16 để lưu (chuẩn WAV)
audio_out_int16 = (audio_eq * 32767).astype(np.int16)

sf.write(OUTPUT_FILE, audio_out_int16, fs)

print(f'Đã lưu: {OUTPUT_FILE}')
print(f'Tần số lấy mẫu: {fs} Hz')
print(f'Số mẫu: {len(audio_out_int16):,}')
print(f'Thời lượng: {len(audio_out_int16)/fs:.2f} giây')
print(f'Định dạng: 16-bit PCM WAV')

# Nghe lại trong Jupyter (nếu hỗ trợ)
try:
    from IPython.display import Audio, display
    print('\n--- Nghe tín hiệu GỐC ---')
    display(Audio(audio_orig, rate=fs))
    print('--- Nghe tín hiệu SAU EQ ---')
    display(Audio(audio_eq, rate=fs))
except:
    print('Không thể phát audio trong notebook này.')
    print(f'Mở file {OUTPUT_FILE} để nghe.')

## 11. Tóm Tắt Kết Quả

In [ ]:
print('=' * 55)
print('   TÓM TẮT KẾT QUẢ EQUALIZER')
print('=' * 55)
print(f'File audio đầu vào : {AUDIO_FILE}')
print(f'File audio đầu ra  : {OUTPUT_FILE}')
print(f'Tần số lấy mẫu    : {fs} Hz')
print(f'Phương pháp lọc   : FIR Bandpass (Hamming window)')
print(f'Số taps (N)        : {NUMTAPS}')
print(f'Bậc lọc (M)        : {NUMTAPS - 1}')
print(f'Độ trễ bộ lọc      : {(NUMTAPS-1)/2/fs*1000:.1f} ms')
print(f'Số bands           : {len(BANDS)}')
print()
print(f'Tín hiệu gốc — RMS : {np.sqrt(np.mean(audio_orig**2)):.4f}')
print(f'Tín hiệu EQ  — RMS : {np.sqrt(np.mean(audio_eq**2)):.4f}')
print()
print('Cài đặt EQ:')
for i, ((name, low, high), gain) in enumerate(zip(BANDS, GAINS_DB)):
    print(f'  Band {i+1:2d} {name:12s}: {gain:+5.1f} dB')
print()
print('Các file đầu ra:')
print('  audio_equalized.wav        — File audio đã cân bằng')
print('  frequency_response.png     — Đáp ứng tần số 10 bands')
print('  waveform_comparison.png    — So sánh waveform')
print('  spectrum_comparison.png    — So sánh phổ tần số')
print('  band_energy.png            — Năng lượng từng band')